In [1]:
import sys,os

fpath = os.path.dirname(os.path.abspath("C:/Users/au646858/OneDrive - Aarhus Universitet/Documents/_repositories/sea-anemone/app/session.py"))
if fpath not in sys.path:
    sys.path.append(fpath)

import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go

In [2]:
from map import Map
from session import Session

In [4]:
sesh = Session.from_workbench("C:\\Users\\au646858\\OneDrive - Aarhus universitet\\Documents\\_repositories\\sea-anemone\\app\\Models\\Surround_MOD_dat.xyz")
sesh.update_site_table()
df = sesh.site.copy()

In [5]:
d_top = sesh.models['top_depth'].unique()
d_bot = sesh.models['bottom_depth'].unique()
res_grid = sesh.models.pivot_table('resistivity', 'site_id', 'center_depth')

print(res_grid.shape)
df=df[df.site_id.isin(res_grid.index)]

(6513, 30)


In [34]:
import numpy as np

def rediscretize(z_old, z_new):
    """this function rediscretizes values onto a new grid. if an interval on the new discretization spans several layers, the resulting average will be thickness averaged."""
    if type(z_old) is tuple:
        z_old1,z_old2 = z_old
    elif isinstance(z_old, np.ndarray):
        z_old1 = np.insert(z_old, 0,0)[:-1]
        z_old2 = z_old*1

    if type(z_new) is tuple:
        z_new1,z_new2 = z_new
    elif isinstance(z_new, np.ndarray):
        z_new1 = np.insert(z_new, 0,0)[:-1]
        z_new2 = z_new*1

    zo1,zn1 = np.meshgrid(z_old1, z_new1, indexing='ij')
    zo2,zn2 = np.meshgrid(z_old2, z_new2, indexing='ij')

    ov1 = np.max(np.stack([zo1,zn1]), axis=0)
    ov2 = np.min(np.stack([zo2,zn2]), axis=0)
    ov = ov2-ov1
    ov[ov<0] = 0

    if z_new2[-1] > z_old2[-1]:
        ov[-1, -1] = 1.
    
    ov /= ov.sum(0, keepdims=True)
    
    return ov

In [35]:
import matplotlib.pyplot as plt

z_old = np.unique(np.concatenate((d_top[1:],d_bot)))
z_old.sort()

z_new = np.array([5., 10., 15., 20., 25., 30., 40., 50., 60., 70., 80., 100., 110., 120., 130., 140., 150., 175., 200., 225. , 250., 300., 350., 400., 450., 500.])
# z_new = np.array([10., 15.])

redisc_mat = rediscretize(z_old, z_new)
print(redisc_mat.shape)
print(res_grid.values.T.shape)
df['mean_res'] = (res_grid.values @ redisc_mat.squeeze())

(30, 26)
(30, 6513)


In [ ]:
df['mean_res']

In [18]:
from model import ModelSection

m2 = ModelSection(dataframe = sesh.models)
m2.render()

In [19]:
def add_scalebar(fig, lat, lon, distance_km=10, n_segments=4, bar_width=6, height_factor=0.15):
    km_per_deg_lon = 111.32 * np.cos(np.radians(lat))
    total_dlon = distance_km / km_per_deg_lon
    seg_dlon = total_dlon / n_segments
    seg_km = distance_km / n_segments
    dlat = total_dlon * height_factor

    colors = ["black", "white"]
    group = "scalebar"
    first = True

    def _add(trace):
        nonlocal first
        trace.legendgroup = group
        if first:
            trace.showlegend = True
            trace.name = "Scale"
            first = False
        else:
            trace.showlegend = False
        fig.add_trace(trace)

    for i in range(n_segments):
        seg_lon0 = lon + i * seg_dlon
        seg_lon1 = lon + (i + 1) * seg_dlon
        _add(go.Scattermap(
            lat=[lat, lat],
            lon=[seg_lon0, seg_lon1],
            mode="lines",
            line=dict(width=bar_width, color=colors[i % 2]),
            hoverinfo="skip",
        ))

    for dlat_offset in [-bar_width * 0.0001, bar_width * 0.0001]:
        _add(go.Scattermap(
            lat=[lat + dlat_offset, lat + dlat_offset],
            lon=[lon, lon + total_dlon],
            mode="lines",
            line=dict(width=1, color="black"),
            hoverinfo="skip",
        ))

    for i in range(n_segments + 1):
        tick_lon = lon + i * seg_dlon
        _add(go.Scattermap(
            lat=[lat - dlat, lat + dlat],
            lon=[tick_lon, tick_lon],
            mode="lines",
            line=dict(width=2, color="black"),
            hoverinfo="skip",
        ))

    for i in [0, n_segments]:
        label_lon = lon + i * seg_dlon
        label_km = int(i * seg_km)
        _add(go.Scattermap(
            lat=[lat - dlat * 2.5],
            lon=[label_lon],
            mode="text",
            text=[f"{label_km}"],
            textfont=dict(size=12, color="black"),
            hoverinfo="skip",
        ))

    _add(go.Scattermap(
        lat=[lat - dlat * 2.5],
        lon=[lon + total_dlon + seg_dlon * 0.4],
        mode="text",
        text=["km"],
        textfont=dict(size=12, color="black"),
        hoverinfo="skip",
    ))

In [20]:
from scipy.spatial import cKDTree

points = np.column_stack([df["longitude"].values, df["latitude"].values])
tree = cKDTree(points)

# Hex radius from median nearest-neighbor distance
dists, _ = tree.query(points, k=2)
default_radius = np.median(dists[:, 1]) * 3

margin = 0.01
bbox = [
    points[:, 0].min() - margin, points[:, 0].max() + margin,
    points[:, 1].min() - margin, points[:, 1].max() + margin,
]

def make_hexagon(cx, cy, radius):
    angles = np.linspace(0, 2 * np.pi, 25)[:-1]
    return [[cx + radius * np.cos(a), cy + radius * np.sin(a)] for a in angles]

def clip_to_bisector(poly, center, neighbor):
    """Clip polygon to the half-plane closer to center than neighbor."""
    mid = [(center[0] + neighbor[0]) / 2, (center[1] + neighbor[1]) / 2]
    nx = center[0] - neighbor[0]
    ny = center[1] - neighbor[1]

    def inside(p):
        return (p[0] - mid[0]) * nx + (p[1] - mid[1]) * ny >= 0

    def intersect(p1, p2):
        d1 = (p1[0] - mid[0]) * nx + (p1[1] - mid[1]) * ny
        d2 = (p2[0] - mid[0]) * nx + (p2[1] - mid[1]) * ny
        t = d1 / (d1 - d2)
        return [p1[0] + t * (p2[0] - p1[0]), p1[1] + t * (p2[1] - p1[1])]

    clipped = []
    prev = poly[-1]
    for curr in poly:
        if inside(curr):
            if not inside(prev):
                clipped.append(intersect(prev, curr))
            clipped.append(curr)
        elif inside(prev):
            clipped.append(intersect(prev, curr))
        prev = curr
    return clipped

def clip_to_bbox(poly, bbox):
    xmin, xmax, ymin, ymax = bbox
    def clip_edge(poly, inside_fn, intersect_fn):
        if not poly:
            return []
        clipped = []
        prev = poly[-1]
        for curr in poly:
            if inside_fn(curr):
                if not inside_fn(prev):
                    clipped.append(intersect_fn(prev, curr))
                clipped.append(curr)
            elif inside_fn(prev):
                clipped.append(intersect_fn(prev, curr))
            prev = curr
        return clipped
    poly = clip_edge(poly, lambda p: p[0] >= xmin,
        lambda p1, p2: [xmin, p1[1] + (p2[1]-p1[1])*(xmin-p1[0])/(p2[0]-p1[0])])
    poly = clip_edge(poly, lambda p: p[0] <= xmax,
        lambda p1, p2: [xmax, p1[1] + (p2[1]-p1[1])*(xmax-p1[0])/(p2[0]-p1[0])])
    poly = clip_edge(poly, lambda p: p[1] >= ymin,
        lambda p1, p2: [p1[0] + (p2[0]-p1[0])*(ymin-p1[1])/(p2[1]-p1[1]), ymin])
    poly = clip_edge(poly, lambda p: p[1] <= ymax,
        lambda p1, p2: [p1[0] + (p2[0]-p1[0])*(ymax-p1[1])/(p2[1]-p1[1]), ymax])
    return poly

# Find neighbors within reach
neighbor_indices = tree.query_ball_point(points, r=default_radius * 3)

features = []
ids = []
z_values = []

for idx in range(len(points)):
    hex_verts = make_hexagon(points[idx, 0], points[idx, 1], default_radius)

    # Clip against perpendicular bisector with each neighbor
    for nb_idx in neighbor_indices[idx]:
        if nb_idx == idx:
            continue
        hex_verts = clip_to_bisector(hex_verts, points[idx], points[nb_idx])
        if len(hex_verts) < 3:
            break

    hex_verts = clip_to_bbox(hex_verts, bbox)

    if len(hex_verts) < 3:
        continue

    coords = [[v[0], v[1]] for v in hex_verts]
    coords.append(coords[0])

    cell_id = str(idx)
    ids.append(cell_id)
    z_values.append(float(df["doi"].iloc[idx]))
    features.append({
        "type": "Feature",
        "id": cell_id,
        "geometry": {"type": "Polygon", "coordinates": [coords]}
    })

geojson = {"type": "FeatureCollection", "features": features}

In [15]:
import geopandas as gpd
from shapely.geometry import MultiPoint, Point, Polygon
from shapely.ops import unary_union
from scipy.spatial import cKDTree

points_xy = np.column_stack([df["longitude"].values, df["latitude"].values])
z_data = df["mean_res"].values

# --- 1. Buffer around all data points ---
tree = cKDTree(points_xy)
nn_dists, _ = tree.query(points_xy, k=2)
cell_size = np.median(nn_dists[:, 1])
buffer_dist = cell_size * 2

mp = MultiPoint([Point(p) for p in points_xy])
buffered = mp.buffer(buffer_dist, resolution=16)

# --- 2. Generate hexagonal mesh inside the buffer ---
xmin, ymin, xmax, ymax = buffered.bounds
dx = cell_size
dy = cell_size * np.sqrt(3) / 2

mesh_points = []
row = 0
y = ymin
while y <= ymax:
    x = xmin + (dx / 2 if row % 2 else 0)
    while x <= xmax:
        p = Point(x, y)
        if buffered.contains(p):
            mesh_points.append([x, y])
        x += dx
    y += dy
    row += 1

mesh_points = np.array(mesh_points)
print(f"Generated {len(mesh_points)} mesh points inside buffer")

# --- 3. IDW interpolation ---
def idw_interpolate(mesh_pts, data_pts, data_vals, power=2, k=12):
    tree = cKDTree(data_pts)
    dists, idxs = tree.query(mesh_pts, k=min(k, len(data_pts)))
    dists = np.maximum(dists, 1e-12)
    weights = 1.0 / dists**power
    weighted_vals = np.sum(weights * data_vals[idxs], axis=1)
    return weighted_vals / np.sum(weights, axis=1)

z_mesh = idw_interpolate(mesh_points, points_xy, z_data, k=7)

# --- 4. Build hex cells with bisector clipping ---
# Start with oversized hex — bisectors will trim to correct Voronoi-like tiling
hex_radius = cell_size * 1.2
hex_angles = np.linspace(0, 2 * np.pi, 7)[:-1]

def clip_to_bisector(poly_coords, center, neighbor):
    mid = [(center[0] + neighbor[0]) / 2, (center[1] + neighbor[1]) / 2]
    nx = center[0] - neighbor[0]
    ny = center[1] - neighbor[1]

    def inside(p):
        return (p[0] - mid[0]) * nx + (p[1] - mid[1]) * ny >= 0

    def intersect(p1, p2):
        d1 = (p1[0] - mid[0]) * nx + (p1[1] - mid[1]) * ny
        d2 = (p2[0] - mid[0]) * nx + (p2[1] - mid[1]) * ny
        t = d1 / (d1 - d2)
        return [p1[0] + t * (p2[0] - p1[0]), p1[1] + t * (p2[1] - p1[1])]

    clipped = []
    prev = poly_coords[-1]
    for curr in poly_coords:
        if inside(curr):
            if not inside(prev):
                clipped.append(intersect(prev, curr))
            clipped.append(curr)
        elif inside(prev):
            clipped.append(intersect(prev, curr))
        prev = curr
    return clipped

# Search all neighbors within reach of the oversized hex
mesh_tree = cKDTree(mesh_points)
neighbor_radius = hex_radius * 2.5
mesh_neighbors = mesh_tree.query_ball_point(mesh_points, r=neighbor_radius)

features = []
ids = []
z_values = []

for i, (cx, cy) in enumerate(mesh_points):
    coords = [[cx + hex_radius * np.cos(a), cy + hex_radius * np.sin(a)]
              for a in hex_angles]

    for nb_idx in mesh_neighbors[i]:
        if nb_idx == i:
            continue
        coords = clip_to_bisector(coords, [cx, cy], list(mesh_points[nb_idx]))
        if len(coords) < 3:
            break

    # if len(coords) < 3:
    #     continue

    # cell_poly = Polygon(coords)
    # clipped = cell_poly.intersection(buffered)

    # if clipped.is_empty or clipped.area < 1e-15:
    #     continue
    # if clipped.geom_type == 'MultiPolygon':
    #     clipped = max(clipped.geoms, key=lambda g: g.area)
    # if clipped.geom_type != 'Polygon':
    #     continue

    # ext_coords = list(clipped.exterior.coords)
    ext_coords = coords

    cell_id = str(i)
    ids.append(cell_id)
    z_values.append(float(z_mesh[i]))
    features.append({
        "type": "Feature",
        "id": cell_id,
        "geometry": {
            "type": "Polygon",
            "coordinates": [[[c[0], c[1]] for c in ext_coords]]
        }
    })

geojson = {"type": "FeatureCollection", "features": features}
print(f"Built {len(features)} hex cells")

KeyboardInterrupt: 

In [25]:
points_xy = np.column_stack([df["longitude"].values, df["latitude"].values])
z_data = df["mean_res"].values

zone_number = sesh.data['zone_number'].iloc[0]
zone_letter = sesh.data['zone_letter'].iloc[0]
import utm
# --- Square pixel covering of dense data points ---
import numpy as np
from scipy.spatial import cKDTree

# Use UTM coordinates from df
data_points = np.column_stack([df["utm_x"].values, df["utm_y"].values])

# 1. Compute median nearest-neighbor distance for pixel size
kdtree = cKDTree(data_points)
nn_dists, _ = kdtree.query(data_points, k=2)
pixel_size = 25.

# 2. Compute bounding box with margin for 4 pixels
margin = 4 * pixel_size
xmin, ymin = data_points.min(axis=0) - margin
xmax, ymax = data_points.max(axis=0) + margin

# 3. Generate grid of pixel centers
nx = int(np.ceil((xmax - xmin) / pixel_size)) + 1
ny = int(np.ceil((ymax - ymin) / pixel_size)) + 1
x_centers = np.linspace(xmin, xmax, nx)
y_centers = np.linspace(ymin, ymax, ny)
xx, yy = np.meshgrid(x_centers, y_centers)
pixel_centers = np.column_stack([xx.ravel(), yy.ravel()])

# 4. For each pixel, check if within 4 pixels of any data point
# (distance <= 4 * pixel_size)
dists, _ = kdtree.query(pixel_centers, k=1)
mask = dists <= 4 * pixel_size
covered_pixels = pixel_centers[mask]
print(f"Generated {len(covered_pixels)} pixels covering data points")
mesh_points = covered_pixels

Generated 6766 pixels covering data points


In [26]:
lat,lon = utm.to_latlon(mesh_points[:, 0], mesh_points[:, 1], zone_number, zone_letter)
mesh_points = np.column_stack([lon, lat])

In [27]:
dx = np.diff(np.unique(mesh_points[:, 0])).mean()
dy = np.diff(np.unique(mesh_points[:, 1])).mean()
hex_radius = np.sqrt((dx**2 + dy**2)/2)

In [28]:
from torch import dist




tree = cKDTree(points_xy)
tree1 = cKDTree(mesh_points)
dists, _ = tree1.query(mesh_points, k=2)
dists = dists[dists>0]

hex_radius = np.min(dists)/(2**0.5)
print(hex_radius)

dists, idxs = tree.query(mesh_points, k=6)

weight = 1./dists
weight /= weight.sum(axis=1, keepdims=True)

z_mesh = weight[:, 0] * z_data[idxs[:, 0]]
for i in range(1, idxs.shape[1]):
    z_mesh += weight[:, i] * z_data[idxs[:, i]]

# dists, idxs = tree.query(mesh_points, k=1)
# z_mesh = z_data[idxs]
z_mesh = np.log10(z_mesh)

# build hexagon features clipped by buffer (keep only centers inside buffered)
hex_angles = np.linspace(0, 2 * np.pi, 5)[:-1]+np.pi/4.
features = []
ids = []
z_values = []

for i, (cx, cy) in enumerate(mesh_points):
    coords = [[cx + hex_radius * np.cos(a), cy + hex_radius * np.sin(a)] for a in hex_angles]
    coords.append(coords[0])
    cell_id = str(i)
    ids.append(cell_id)
    z_values.append(float(z_mesh[i]))
    features.append({
        "type": "Feature",
        "id": cell_id,
        "geometry": {
            "type": "Polygon",
            "coordinates": [coords]
        }
    })

geojson = {"type": "FeatureCollection", "features": features}
print(f"Built {len(features)} hex cells (regular 100x100 grid)")

0.0001586430727738259
Built 6766 hex cells (regular 100x100 grid)


In [39]:
class Grid:
    def __init__(self, lat, lon, pixel_size=25., buffer_pixel=4):
        self.lat = lat
        self.lon = lon
        utm_x, utm_y, zone, letter = utm.from_latlon(lat.mean(), lon.mean())
        self.utm_x, self.utm_y, _, __ = utm.from_latlon(lat, lon, force_zone_number=zone, force_zone_letter=letter)
        self.zone, self.letter = zone, letter
        self.points_latlon = np.column_stack((self.lon, self.lat))
        self.points_xy = np.column_stack((self.utm_x, self.utm_y))
        self.pixel_size = pixel_size
        self.buffer_pixel = buffer_pixel
        self.calculate_grid()

    def calculate_grid(self):
        from scipy.spatial import cKDTree

        kdtree = cKDTree(self.points_xy)
        nn_dists, _ = kdtree.query(self.points_xy, k=2)
        self.pixel_size = self.pixel_size

        # 2. Compute bounding box with margin for 4 pixels
        margin = self.buffer_pixel * self.pixel_size
        xmin, ymin = self.points_xy.min(axis=0) - margin
        xmax, ymax = self.points_xy.max(axis=0) + margin

        # 3. Generate grid of pixel centers
        nx = int(np.ceil((xmax - xmin) / self.pixel_size)) + 1
        ny = int(np.ceil((ymax - ymin) / self.pixel_size)) + 1
        x_centers = np.linspace(xmin, xmax, nx)
        y_centers = np.linspace(ymin, ymax, ny)
        xx, yy = np.meshgrid(x_centers, y_centers)
        pixel_centers = np.column_stack([xx.ravel(), yy.ravel()])


        dists, _ = kdtree.query(pixel_centers, k=1)
        mask = dists <= margin
        covered_pixels = pixel_centers[mask]
        mesh_points = covered_pixels

        coordinates_x = mesh_points[:, 0][:, None] + np.array([-1,  1, 1, -1, -1])[None,:]*self.pixel_size/2.
        coordinates_y = mesh_points[:, 1][:, None] + np.array([-1, -1, 1,  1, -1])[None,:]*self.pixel_size/2.

        clat, clon = utm.to_latlon(mesh_points[:, 0], mesh_points[:, 1], self.zone, self.letter)
        self.mesh_points = np.column_stack([clon, clat])

        coordinates_lat, coordinates_lon = utm.to_latlon(coordinates_x, coordinates_y, self.zone, self.letter)

        tree = cKDTree(self.points_xy)
        self.mesh_dists, self.mesh_idxs = tree.query(self.mesh_points, k=6)
        features = []
        ids = []

        for i, (cx, cy) in enumerate(mesh_points):
            coords = [(x,y) for x,y in zip(coordinates_lon[i], coordinates_lat[i])]
            cell_id = str(i)
            ids.append(cell_id)
            features.append({
                "type": "Feature",
                "id": cell_id,
                "geometry": {
                    "type": "Polygon",
                    "coordinates": [coords]
                }
            })
        self.geojson = {"type": "FeatureCollection", "features": features}
        self.ids = ids
        print('n_pixels:', len(self.mesh_points))

    def grid_values(self, z_data):
        weight = 1./self.mesh_dists
        weight /= weight.sum(axis=1, keepdims=True)
        z_mesh = weight[:, 0] * z_data[self.mesh_idxs[:, 0]]
        for i in range(1, self.mesh_idxs.shape[1]):
            z_mesh += weight[:, i] * z_data[self.mesh_idxs[:, i]]
        return z_mesh
    
    def graph_object(self, z_data, **kwargs):
        z_mesh = self.grid_values(z_data)
        opacity = kwargs.pop('opacity', 1)
        return go.Choroplethmap(
            geojson=self.geojson,
            locations=self.ids,
            z=z_mesh,
            showlegend=True,
            marker=dict(opacity=opacity, line=dict(width=0, color=None)),
            showscale=True,
            hoverinfo="skip",
            selectedpoints=[],
            unselected=dict(marker=dict(opacity=opacity)),            
            **kwargs
        )


In [40]:
g1 = Grid(df.latitude.values, df.longitude.values, pixel_size=25., buffer_pixel=4)

n_pixels: 6766


In [41]:
# m1 = Map(data=sesh.site)

m1.settings.markersize=5.
m1.settings.plot_by = 'misfit'
m1.settings.dragmode='pan'

fig = m1.render()

# fig.add_trace(go.Choroplethmap(
#     geojson=geojson,
#     locations=ids,
#     z=z_values,
#     colorscale="Turbo",
#     marker=dict(opacity=1, line=dict(width=0, color=None)),
#     showscale=True,
#     hoverinfo="skip",
#     selectedpoints=[],
#     unselected=dict(marker=dict(opacity=1)),
#     name="Mean Res.",   # legend label
#     showlegend=True              # ensure it appears in the legend
# ))

fig.add_trace(g1.graph_object(df["mean_res"].values, colorscale="Turbo", name="Mean Res.", opacity=0.8))

# fig.update_layout(
#     clickmode="event",       # don't select on click
# )
data_list = list(fig.data)
fig.data = [data_list[-1]] + data_list[:-1]
bar_lat = df["latitude"].min() - 0.005
bar_lon = df["longitude"].min()

add_scalebar(fig, bar_lat, bar_lon, distance_km=3, n_segments=3, height_factor=0.01)
fig.update_layout(showlegend=True,
    legend=dict(
        x=0.01,
        y=0.99,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.5)",  # semi-transparent background
        bordercolor="black",
        borderwidth=1,
    )
)
fig